# Caso MPG

Breve descripción del dataset 'MPG' y sus columnas:

El conjunto de datos MPG contiene registros de automóviles con medidas relacionadas con el consumo de combustible y características del vehículo. Es útil para explorar cómo las propiedades del vehículo afectan la eficiencia de combustible.

Columnas principales (explicación muy simple):
- **mpg:** consumo de combustible (millas por galón).
- **cylinders:** número de cilindros del motor.
- **displacement:** tamaño del motor (desplazamiento).
- **horsepower:** potencia del motor (caballos de fuerza).
- **weight:** peso del vehículo.
- **acceleration:** medida de aceleración del vehículo.
- **model_year:** año del modelo del vehículo.
- **origin:** región de origen del vehículo (por ejemplo: USA, Europe, Japan).
- **name:** nombre o modelo del vehículo.

Uso sugerido: usa estas columnas para predecir 'mpg' y para análisis exploratorio sencillo (correlaciones y visualizaciones).

## Extracción de Datos

In [1]:
import requests, sqlite3, pandas as pd

url = "https://raw.githubusercontent.com/davidjamesknight/SQLite_databases_for_learning_data_science/main/mpg.db"
r = requests.get(url)

with open("mpg.db", "wb") as f:
    f.write(r.content)

conn = sqlite3.connect("mpg.db")

query = """
SELECT
    O.mpg,
    O.cylinders,
    O.displacement,
    O.horsepower,
    O.weight,
    O.acceleration,
    O.model_year,
    ORG.origin,
    N.name
FROM
    Observation AS O
JOIN
    Origin AS ORG ON O.origin_id = ORG.origin_id
JOIN
    Name AS N ON O.name_id = N.name_id
"""

df = pd.read_sql_query(query, conn)
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,18.0,8,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693,11.5,70,usa,buick skylark 320
2,18.0,8,318.0,150.0,3436,11.0,70,usa,plymouth satellite
3,16.0,8,304.0,150.0,3433,12.0,70,usa,amc rebel sst
4,17.0,8,302.0,140.0,3449,10.5,70,usa,ford torino


## Exploración de Datos

In [2]:
# Identificar tipos de datos
df.dtypes.to_frame("dtype")

,dtype
mpg,float64
cylinders,int64
displacement,float64
horsepower,float64
weight,int64
acceleration,float64
model_year,int64
origin,str
name,str


In [3]:
# Detectar nulos
null_counts = df.isna().sum()
null_pct = (null_counts / len(df) * 100).round(2)

null_report = pd.DataFrame({
    "null_count": null_counts,
    "null_pct": null_pct
}).sort_values("null_pct", ascending=False)

null_report[null_report["null_count"] > 0]

,null_count,null_pct
horsepower,6,1.51


In [4]:
# Tratar nulos
rows_before = len(df)

# En este dataset, horsepower puede venir como texto o contener valores no numéricos
df["horsepower"] = pd.to_numeric(df["horsepower"], errors="coerce")

null_counts = df.isna().sum()
null_pct = (null_counts / len(df) * 100)

cols_with_nulls = null_counts[null_counts > 0].index.tolist()
cols_over_5 = [c for c in cols_with_nulls if null_pct[c] > 5]
cols_5_or_less = [c for c in cols_with_nulls if null_pct[c] <= 5]

# >5%: imputar; <=5%: eliminar filas con nulos
for col in cols_over_5:
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].fillna(df[col].median())
    else:
        df[col] = df[col].fillna(df[col].mode().iloc[0])

if cols_5_or_less:
    df = df.dropna(subset=cols_5_or_less).copy()

rows_after = len(df)
null_treatment_summary = {
    "rows_before": rows_before,
    "rows_after": rows_after,
    "rows_removed": rows_before - rows_after,
    "imputed_cols": cols_over_5,
    "dropped_rows_on_cols": cols_5_or_less
}

null_treatment_summary

{'rows_before': 398,
 'rows_after': 392,
 'rows_removed': 6,
 'imputed_cols': [],
 'dropped_rows_on_cols': ['horsepower']}

In [5]:
# ¿Se deben conservar variables con nulos?
if null_treatment_summary["imputed_cols"]:
    print("Se conservaron columnas con nulos >5% imputando valores:")
    print(null_treatment_summary["imputed_cols"])
else:
    print("No hubo columnas con más de 5% de nulos para imputar.")

if null_treatment_summary["dropped_rows_on_cols"]:
    print("Se eliminaron filas en columnas con <=5% de nulos:")
    print(null_treatment_summary["dropped_rows_on_cols"])
    print(f"Filas eliminadas: {null_treatment_summary['rows_removed']}")
else:
    print("No fue necesario eliminar filas por nulos <=5%.")

df.isna().sum().sum()

No hubo columnas con más de 5% de nulos para imputar.
Se eliminaron filas en columnas con <=5% de nulos:
['horsepower']
Filas eliminadas: 6


np.int64(0)

In [6]:
# Resumen estadístico del dataset
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
mpg,392.0,NaN,NaN,NaN,23.445918,7.805007,9.0,17.0,22.75,29.0,46.6
cylinders,392.0,NaN,NaN,NaN,5.471939,1.705783,3.0,4.0,4.0,8.0,8.0
displacement,392.0,NaN,NaN,NaN,194.41199,104.644004,68.0,105.0,151.0,275.75,455.0
horsepower,392.0,NaN,NaN,NaN,104.469388,38.49116,46.0,75.0,93.5,126.0,230.0
weight,392.0,NaN,NaN,NaN,2977.584184,849.40256,1613.0,2225.25,2803.5,3614.75,5140.0
acceleration,392.0,NaN,NaN,NaN,15.541327,2.758864,8.0,13.775,15.5,17.025,24.8
model_year,392.0,NaN,NaN,NaN,75.979592,3.683737,70.0,73.0,76.0,79.0,82.0
origin,392,3,usa,245,NaN,NaN,NaN,NaN,NaN,NaN,NaN
name,392,301,amc matador,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Distribuciones Numéricas

In [7]:
# Visualización de distribuciones para variables numéricas con Plotly
import plotly.express as px

numerical_vars = df.select_dtypes(include="number").columns.tolist()
numerical_vars

for col in numerical_vars:
    fig = px.histogram(
        df,
        x=col,
        marginal="box",
        nbins=30,
        title=f"Distribución de {col}"
    )
    fig.show()

### Frecuencias Categóricas

In [8]:
# Visualización de frecuencias de variables categóricas con Plotly
cat_vars = df.select_dtypes(exclude="number").columns.tolist()
cat_vars

for col in cat_vars:
    freq = df[col].value_counts().reset_index()
    freq.columns = [col, "count"]

    fig = px.bar(
        freq,
        x=col,
        y="count",
        title=f"Frecuencias de {col}",
        text="count"
    )
    fig.show()

### Matriz de Correlación

In [9]:
# Crear matriz de correlación en variables numéricas
import plotly.figure_factory as ff

corr_matrix = df[numerical_vars].corr(numeric_only=True).round(2)

fig = ff.create_annotated_heatmap(
    z=corr_matrix.values,
    x=list(corr_matrix.columns),
    y=list(corr_matrix.index),
    annotation_text=corr_matrix.values,
    colorscale="RdBu",
    zmin=-1,
    zmax=1,
    showscale=True
)
fig.update_layout(title="Matriz de Correlación (Variables Numéricas)")
fig.show()

In [10]:
# Crear un pairplot con Plotly con proporciones ajustadas
# Este va de regalo

import plotly.express as px

fig = px.scatter_matrix(
    df,
    dimensions=numerical_vars,
    color='origin',
    title='Pairplot de Variables Numéricas',
    labels={col: col.capitalize() for col in numerical_vars}
)
fig.update_layout(
    width=1200,
    height=1200,
    title_font_size=20
)
fig.update_traces(diagonal_visible=True)
fig.show()

### Conclusiones del analisis exploratorio

- `weight`, `horsepower` y `displacement` suelen mostrar relacion inversa con `mpg` (autos mas pesados/potentes, menor eficiencia).
- `cylinders` tiende a moverse en la misma direccion que `weight` y `displacement`, por lo que puede existir multicolinealidad.
- `origin` puede aportar senal util al modelo y debe codificarse para incluirse en regresion.
- Tras el tratamiento de nulos, el dataset queda listo para preprocesamiento y entrenamiento.

## Preprocesamiento

In [11]:
# Identificar variables independientes (X) y dependiente (y)
# NO incluir la variable 'name'

X = df.drop(columns=["mpg", "name"]).copy()
y = df["mpg"].copy()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
X.head()

X shape: (392, 7)
y shape: (392,)


,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
0,8,307.0,130.0,3504,12.0,70,usa
1,8,350.0,165.0,3693,11.5,70,usa
2,8,318.0,150.0,3436,11.0,70,usa
3,8,304.0,150.0,3433,12.0,70,usa
4,8,302.0,140.0,3449,10.5,70,usa


In [12]:
# Train/test split 80/20 con random_state=42
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"y_train: {y_train.shape} | y_test: {y_test.shape}")

X_train: (313, 7) | X_test: (79, 7)
y_train: (313,) | y_test: (79,)


In [13]:
# One Hot Encoder a 'origin'
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

X_train_origin_ohe = ohe.fit_transform(X_train[["origin"]])
X_test_origin_ohe = ohe.transform(X_test[["origin"]])

origin_cols = ohe.get_feature_names_out(["origin"]).tolist()

X_train_origin_df = pd.DataFrame(
    X_train_origin_ohe, columns=origin_cols, index=X_train.index
)
X_test_origin_df = pd.DataFrame(
    X_test_origin_ohe, columns=origin_cols, index=X_test.index
)

origin_cols

['origin_europe', 'origin_japan', 'origin_usa']

In [14]:
# Combinar variable categórica 'origin' codificada con datasets originales
X_train_final = pd.concat([X_train.drop(columns=["origin"]), X_train_origin_df], axis=1)
X_test_final = pd.concat([X_test.drop(columns=["origin"]), X_test_origin_df], axis=1)

print(f"X_train_final: {X_train_final.shape} | X_test_final: {X_test_final.shape}")
X_train_final.head()

X_train_final: (313, 9) | X_test_final: (79, 9)


,cylinders,displacement,horsepower,weight,acceleration,model_year,origin_europe,origin_japan,origin_usa
260,6,225.0,110.0,3620,18.7,78,0.0,0.0,1.0
184,4,140.0,92.0,2572,14.9,76,0.0,0.0,1.0
174,6,171.0,97.0,2984,14.5,75,0.0,0.0,1.0
64,8,318.0,150.0,4135,13.5,72,0.0,0.0,1.0
344,4,86.0,64.0,1875,16.4,81,0.0,0.0,1.0


## Ajustar Modelo

In [15]:
# Entrenar y evaluar modelo simple: mpg vs variable más correlacionada
import statsmodels.api as sm

corr_train = pd.concat([X_train_final, y_train], axis=1).corr(numeric_only=True)
best_feature = corr_train["mpg"].drop("mpg").abs().sort_values(ascending=False).index[0]

print(f"Variable más correlacionada con mpg: {best_feature}")

X_train_simple = sm.add_constant(X_train_final[[best_feature]], has_constant="add")
simple_model = sm.OLS(y_train, X_train_simple).fit()

print(simple_model.summary())

Variable más correlacionada con mpg: weight
                            OLS Regression Results                            
Dep. Variable:                    mpg   R-squared:                       0.698
Model:                            OLS   Adj. R-squared:                  0.697
Method:                 Least Squares   F-statistic:                     719.4
Date:                Sun, 08 Mar 2026   Prob (F-statistic):           6.83e-83
Time:                        20:39:59   Log-Likelihood:                -905.30
No. Observations:                 313   AIC:                             1815.
Df Residuals:                     311   BIC:                             1822.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const   

In [16]:
# Calcular R² y RMSE en test para el modelo simple
from sklearn.metrics import mean_squared_error, r2_score

X_test_simple = sm.add_constant(X_test_final[[best_feature]], has_constant="add")
y_pred_simple = simple_model.predict(X_test_simple)

r2_simple = r2_score(y_test, y_pred_simple)
rmse_simple = mean_squared_error(y_test, y_pred_simple) ** 0.5

print(f"Modelo simple ({best_feature})")
print(f"R2 test  : {r2_simple:.4f}")
print(f"RMSE test: {rmse_simple:.4f}")

Modelo simple (weight)
R2 test  : 0.6533
RMSE test: 4.2064


In [17]:
# Ahora con TODAS las variables
import statsmodels.api as sm

X_train_full = sm.add_constant(X_train_final, has_constant="add")
full_model = sm.OLS(y_train, X_train_full).fit()

print(full_model.summary())

                            OLS Regression Results                            
Dep. Variable:                    mpg   R-squared:                       0.829
Model:                            OLS   Adj. R-squared:                  0.824
Method:                 Least Squares   F-statistic:                     183.8
Date:                Sun, 08 Mar 2026   Prob (F-statistic):          1.20e-111
Time:                        20:40:00   Log-Likelihood:                -816.67
No. Observations:                 313   AIC:                             1651.
Df Residuals:                     304   BIC:                             1685.
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const           -12.9786      4.049     -3.205

In [18]:
# Calcular R² y RMSE en test para el modelo con todas las variables
X_test_full = sm.add_constant(X_test_final, has_constant="add")
y_pred_full = full_model.predict(X_test_full)

r2_full = r2_score(y_test, y_pred_full)
rmse_full = mean_squared_error(y_test, y_pred_full) ** 0.5

print("Modelo completo")
print(f"R2 test  : {r2_full:.4f}")
print(f"RMSE test: {rmse_full:.4f}")

comparison = pd.DataFrame({
    "modelo": [f"Simple ({best_feature})", "Completo"],
    "R2_test": [r2_simple, r2_full],
    "RMSE_test": [rmse_simple, rmse_full]
})
comparison

Modelo completo
R2 test  : 0.7923
RMSE test: 3.2561


,modelo,R2_test,RMSE_test
0,Simple (weight),0.653347,4.206351
1,Completo,0.792277,3.256114


💡 En Machine Learning puro se prioriza R². En econometría y ciencias sociales se prioriza la validez estadística. Ambos mundos son válidos, pero tienen distintos criterios de éxito.